# Thread Pool Execter 
- It is part of Python’s concurrent.futures module.
- t manages a pool of threads for you
- Instead of manually creating multiple threading.Thread() objects, you submit tasks to the pool.
- The executor handles thread creation, execution, and joining automatically.

--- 
## syntax

``` python 

###################################################################
#              MAP 
##################################################################
from concurrent.futures import ThreadPoolExecutor  # or ProcessPoolExecutor

with ThreadPoolExecutor(max_workers=N) as executor:
    results = executor.map(function_name, iterable_of_inputs)
    
# results is an iterator
for r in results:
    print(r)


###################################################################
#              SUBMIT
##################################################################

from concurrent.futures import ThreadPoolExecutor

# create a thread pool with max_workers threads
with ThreadPoolExecutor(max_workers=5) as executor:
    # submit tasks to the pool
    future1 = executor.submit(function_name, arg1, arg2)
    future2 = executor.submit(function_name, arg1, arg2)

# wait for all tasks to complete automatically when exiting the 'with' block

# max_workers → maximum number of threads in the pool
# submit() → submit a function (with optional arguments) to run in a thread
# result() → get the return value of the function

```

- submit() runs a single function call in a separate thread
- Returns a Future object → you can check status, wait, or get result
- You can submit tasks dynamically, unlike map where you need all inputs upfront


| Feature      | `executor.map`              | `executor.submit`                                      |
| ------------ | --------------------------- | ------------------------------------------------------ |
| Input        | Iterable of inputs          | Single input per call                                  |
| Submission   | All tasks submitted at once | Can submit dynamically anytime                         |
| Result order | Preserved same as input     | Preserved if you iterate `futures` in submission order |
| Flexibility  | Less flexible               | Very flexible, can check status, cancel, etc.          |


In [1]:
from concurrent.futures import ThreadPoolExecutor
import time

In [3]:
def print_num(num):
    time.sleep(1)
    return f"Number : {num}"

nums = [1,2,3,4,5]

with ThreadPoolExecutor(max_workers=3) as executor:
    results  = executor.map(print_num, nums)

for r in results:
    print(r)
    

Number : 1
Number : 2
Number : 3
Number : 4
Number : 5


| Step | Thread                | Task         |
| ---- | --------------------- | ------------ |
| 1    | Thread-1              | print_num(1) |
| 2    | Thread-2              | print_num(2) |
| 3    | Thread-3              | print_num(3) |
| 4    | Thread-1 (finished 1) | print_num(4) |
| 5    | Thread-2 (finished 2) | print_num(5) |


- Notice: threads are reused when they finish a task
- This is called a thread pool, threads are persist

In [4]:
def power(base, exp):
    time.sleep(1)
    return base ** exp

bases = [2, 3, 4]
exponents = [3, 2, 1]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(power, bases, exponents)

print(list(results))  # [8, 9, 4]

[8, 9, 4]


In [5]:
# using submit

from concurrent.futures import ThreadPoolExecutor


def print_num(num):
    time.sleep(1)
    return f"Number : {num}"

nums = [1, 2, 3, 4, 5]

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(print_num,n) for n in nums]

    for future in futures:
        print(future.result())

Number : 1
Number : 2
Number : 3
Number : 4
Number : 5


In [7]:
# map

import requests
from concurrent.futures import ThreadPoolExecutor

urls = [
    "https://httpbin.org/get",
    "https://jsonplaceholder.typicode.com/posts",
    "https://httpbin.org/uuid",
    "https://jsonplaceholder.typicode.com/comments"
]

def fetch_length(url):
    r = requests.get(url)
    return f"URL: {url} - Length: {len(r.text)}"

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(fetch_length, urls)  # submit all tasks at once

for r in results:
    print(r)

URL: https://httpbin.org/get - Length: 314
URL: https://jsonplaceholder.typicode.com/posts - Length: 27520
URL: https://httpbin.org/uuid - Length: 53
URL: https://jsonplaceholder.typicode.com/comments - Length: 157745


In [9]:
import requests
from concurrent.futures import ThreadPoolExecutor

urls = [
    "https://httpbin.org/get",
    "https://jsonplaceholder.typicode.com/posts",
    "https://httpbin.org/uuid",
    "https://jsonplaceholder.typicode.com/comments"
]

def fetch_length(url):
    r = requests.get(url)
    return f"URL: {url} - Length: {len(r.text)}"

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(fetch_length, url) for url in urls]

    for future in futures:
        print(future.result())

URL: https://httpbin.org/get - Length: 314
URL: https://jsonplaceholder.typicode.com/posts - Length: 27520
URL: https://httpbin.org/uuid - Length: 53
URL: https://jsonplaceholder.typicode.com/comments - Length: 157745


In [ ]:
# above code with list comprehension

import requests
from concurrent.futures import ThreadPoolExecutor

urls = [
    "https://httpbin.org/get",
    "https://jsonplaceholder.typicode.com/posts",
    "https://httpbin.org/uuid",
    "https://jsonplaceholder.typicode.com/comments"
]

def fetch_length(url):
    r = requests.get(url)
    return f"URL: {url} - Length: {len(r.text)}"

# Create thread pool
executor = ThreadPoolExecutor(max_workers=3)

# Step 1: Submit tasks individually
futures = []
for url in urls:
    future = executor.submit(fetch_length, url)
    futures.append(future)

# Step 2: Collect results
for future in futures:
    print(future.result())

# Step 3: Shutdown executor
executor.shutdown()

URL: https://httpbin.org/get - Length: 314
URL: https://jsonplaceholder.typicode.com/posts - Length: 27520
URL: https://httpbin.org/uuid - Length: 53
URL: https://jsonplaceholder.typicode.com/comments - Length: 157745


| Feature                | Manual `threading.Thread()` | ThreadPoolExecutor                    |
| ---------------------- | --------------------------- | ------------------------------------- |
| Thread creation        | Manual                      | Automatic                             |
| Thread reuse           | No                          | Yes (pool)                            |
| Max workers            | Manual control              | `max_workers` param                   |
| Exception handling     | Manual                      | Automatic via `Future.result()`       |
| Task submission        | Fixed code                  | Dynamic submission + flexible control |
| Switching to CPU-bound | Hard                        | Easy: use `ProcessPoolExecutor`       |
